# **Inicialização do projeto**

In [5]:
#Conjunto de Imports para funcionamento de todas as células do notebook
import os
import urllib.request
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import countDistinct
from pyspark.sql.functions import col
import matplotlib.pyplot as plt
from pyspark.sql.functions import min as spark_min, max as spark_max, col
import seaborn as sns
from pyspark.sql.types import DoubleType

In [6]:
#Início da engine para o projeto
spark = SparkSession.builder.appName("eda_taxi") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

destino = Path(os.getcwd()) / "data/1_bronze/yellow_trips/yellow_tripdata_2024-01.parquet"
destino.parent.mkdir(parents=True, exist_ok=True)

df = spark.read.parquet(str(destino))

In [7]:
# Baixa o dicionário de zonas da internet e o carrega como uma tabela Spark

destino_zone = Path(os.getcwd()) / "data/1_bronze/yellow_trips" / "taxi_zone_lookup.csv"
destino_zone.parent.mkdir(parents=True, exist_ok=True)

zone_lookup = spark.read.csv(str(destino_zone), header=True, inferSchema=True)
zone_lookup.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



# **Tratamento**

# **0. Adição de novas colunas**

Para um maior enriquecimento dos dados da camada ouro, assim como para
suprir necessidades existentes no processo de tratamento, optou-se
pela adição das seguintes colunas:

- **`trip_duration_min`** — duração da corrida em minutos, calculada
  pela diferença entre `tpep_dropoff_datetime` e
  `tpep_pickup_datetime`. Serve de base para as demais colunas
  derivadas e para a identificação de durações anômalas (seção 1.9).

- **`average_speed_mph`** — velocidade média implícita da corrida, em
  milhas por hora, calculada como `trip_distance` dividido pela
  duração (`trip_duration_min`). Protegida contra divisão por
  zero/negativo (retorna `NULL` quando `trip_duration_min <= 0`).
  Utilizada como critério central para identificar valores anômalos de
  `trip_distance`, ancorado no limite legal de velocidade do estado de
  Nova York (65 mph).

- **`cost_per_mile`** — custo da corrida por milha percorrida,
  calculado como `fare_amount` dividido por `trip_distance`. Protegida
  contra divisão por zero (retorna `NULL` quando `trip_distance == 0`).
  Permite avaliar a proporcionalidade entre tarifa cobrada e distância
  percorrida, servindo de apoio à investigação de outliers em
  `fare_amount` que a magnitude bruta, isolada, não captura.

In [8]:
# Adicionar colunas extras - trip_duration, average_speed_mpd, cost_per_mile

df = df.withColumn(
    "trip_duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
).withColumn(
    "average_speed_mph",
    F.when(F.col("trip_duration_min") > 0,
           F.col("trip_distance") / (F.col("trip_duration_min") / 60))
     .otherwise(None)
).withColumn(
    "cost_per_mile",
    F.when(F.col("trip_distance") > 0,
           F.col("fare_amount") / F.col("trip_distance"))
     .otherwise(None)
)

df.select("trip_duration_min", "average_speed_mph", "cost_per_mile").show(10)

+------------------+------------------+------------------+
| trip_duration_min| average_speed_mph|     cost_per_mile|
+------------------+------------------+------------------+
|              19.8| 5.212121212121212|10.290697674418604|
|               6.6|16.363636363636363| 5.555555555555555|
|17.916666666666668|15.739534883720932| 4.957446808510638|
|               8.3|10.120481927710843| 7.142857142857143|
|               6.1| 7.868852459016395|             9.875|
| 32.38333333333333| 8.708183221821926| 6.297872340425532|
|             26.05|24.921305182341648|  4.22365988909427|
|              28.0| 6.428571428571429| 8.466666666666667|
|28.183333333333334|11.581312832643407| 5.698529411764706|
|1.1333333333333333|2.1176470588235294|              75.0|
+------------------+------------------+------------------+
only showing top 10 rows



# **1. Valores nulos**

## **1.1. Valores nulos - RatecodeID, passenger_count, store_and_fwd_flag, congestion_surcharge, Airport_fee**

Todos os nulos identificados nestas cinco colunas coincidem em 100%
dos casos com a adoção do Flex Fare como método de pagamento
(`payment_type = 0`). O tratamento aplicado difere pelo tipo da
coluna:

- **`RatecodeID` e `store_and_fwd_flag`** (categóricas): o valor nulo
  é substituído diretamente pelo rótulo **"invalido"**.
- **`passenger_count`, `congestion_surcharge` e `Airport_fee`**
  (numéricas): o valor é **mantido como `NULL`**, preservando o tipo
  numérico da coluna para permitir agregações futuras (médias, somas).
  Em seu lugar, foi criada uma **coluna de status separada**
  (`passenger_count_status`, `congestion_surcharge_status`,
  `Airport_fee_status`), marcada como **"invalido"** para essas linhas. Consultas que
  precisem operar sobre os valores numéricos devem filtrar pelo status
  correspondente antes de agregar.

**Por que o Flex Fare gera esses nulos:** o Flex Fare é um programa da
TLC que oferece tarifa combinada e garantida antes do início da
viagem, como alternativa ao taxímetro tradicional — o mesmo modelo de
preço antecipado já usado por aplicativos de transporte por
aplicativo. Como o preço é acordado previamente, a viagem não passa
pelo ciclo operacional do taxímetro que gera esses campos: `RatecodeID`
representa "o código de tarifa final em vigor ao término da viagem"
(um conceito do cálculo por taxímetro) e `store_and_fwd_flag` indica
se o registro foi retido na memória do veículo por falta de conexão
com o servidor — ambos vinculados à operação do taxímetro convencional,
que o Flex Fare substitui. A nulidade reflete, portanto, uma diferença
estrutural no processo de captura, não uma falha de preenchimento.

**Fontes:**
- Notice of Promulgation — Flex Fare Rule Package (TLC, 14/08/2024):
  https://www.nyc.gov/assets/tlc/downloads/pdf/flex_fare_rule_package_08_14_24.pdf
- Data Dictionary – Yellow Taxi Trip Records (TLC, 18/03/2025):
  https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf

In [9]:
# Colunas categóricas (aceitam texto diretamente)
# Mudando RatecodeID e store_and_fwd_flag para 'invalido' nas células nulas
df = df.withColumn(
    "RatecodeID",
    F.when(F.col("RatecodeID").isNull(), "invalido").otherwise(F.col("RatecodeID").cast("int").cast("string"))
).withColumn(
    "store_and_fwd_flag",
    F.when(F.col("store_and_fwd_flag").isNull(), "invalido").otherwise(F.col("store_and_fwd_flag"))
)

# Colunas numéricas — não aceitam texto sem mudar o tipo da coluna.
# Criando colunas flag para passenger_count, congestion_surcharge e Airport_fee e definindo flag como 'invalido' para as
# células nulas
df = df.withColumn(
    "passenger_count_status",
    F.when(F.col("passenger_count").isNull(), "invalido").otherwise("valido")
).withColumn(
    "congestion_surcharge_status",
    F.when(F.col("congestion_surcharge").isNull(), "invalido").otherwise("valido")
).withColumn(
    "Airport_fee_status",
    F.when(F.col("Airport_fee").isNull(), "invalido").otherwise("valido")
)

# Conferindo dados
for col in ["RatecodeID", "passenger_count", "store_and_fwd_flag", "congestion_surcharge", "Airport_fee"]:
    print(f"\n=== {col} ===")
    df.groupBy(col).count().orderBy(F.desc("count")).show(10)


=== RatecodeID ===
+----------+-------+
|RatecodeID|  count|
+----------+-------+
|         1|2663350|
|  invalido| 140162|
|         2|  98713|
|        99|  28663|
|         5|  19410|
|         3|   7954|
|         4|   6365|
|         6|      7|
+----------+-------+


=== passenger_count ===
+---------------+-------+
|passenger_count|  count|
+---------------+-------+
|              1|2188739|
|              2| 405103|
|           NULL| 140162|
|              3|  91262|
|              4|  51974|
|              5|  33506|
|              0|  31465|
|              6|  22353|
|              8|     51|
|              7|      8|
+---------------+-------+
only showing top 10 rows


=== store_and_fwd_flag ===
+------------------+-------+
|store_and_fwd_flag|  count|
+------------------+-------+
|                 N|2813126|
|          invalido| 140162|
|                 Y|  11336|
+------------------+-------+


=== congestion_surcharge ===
+--------------------+-------+
|congestion_surchar

In [10]:
# Visualizar distribuição de variáveis flag

df.groupBy("passenger_count_status").count().show()

df.groupBy("congestion_surcharge_status").count().show()

df.groupBy("Airport_fee_status").count().show()

# passenger_count_status: string, congestion_surcharge_status: string, Airport_fee_status: string

+----------------------+-------+
|passenger_count_status|  count|
+----------------------+-------+
|                valido|2824462|
|              invalido| 140162|
+----------------------+-------+

+---------------------------+-------+
|congestion_surcharge_status|  count|
+---------------------------+-------+
|                     valido|2824462|
|                   invalido| 140162|
+---------------------------+-------+

+------------------+-------+
|Airport_fee_status|  count|
+------------------+-------+
|            valido|2824462|
|          invalido| 140162|
+------------------+-------+



## **1.2. Valores nulos - average_speed_mph**

Em função da variável average_speed_mph ser numérica, optou-se pela criação de uma coluna de status associada a ela para lidar com os dados nulos.

In [11]:
df = df.withColumn(
    "average_speed_mph_status",
    F.when(F.col("average_speed_mph").isNull(), "invalido").otherwise("valido")
)

In [12]:
df.select("average_speed_mph", "average_speed_mph_status").show()

+------------------+------------------------+
| average_speed_mph|average_speed_mph_status|
+------------------+------------------------+
| 5.212121212121212|                  valido|
|16.363636363636363|                  valido|
|15.739534883720932|                  valido|
|10.120481927710843|                  valido|
| 7.868852459016395|                  valido|
| 8.708183221821926|                  valido|
|24.921305182341648|                  valido|
| 6.428571428571429|                  valido|
|11.581312832643407|                  valido|
|2.1176470588235294|                  valido|
|  7.12401055408971|                  valido|
| 7.955801104972375|                  valido|
|13.442622950819672|                  valido|
| 6.428571428571429|                  valido|
|12.521739130434781|                  valido|
| 21.73913043478261|                  valido|
| 7.142857142857143|                  valido|
|               0.0|                  valido|
|13.533834586466165|              

In [13]:
# Visualizar distribuição da variável flag associada a average_speed_mph

df.groupBy("average_speed_mph_status").count().show()

+------------------------+-------+
|average_speed_mph_status|  count|
+------------------------+-------+
|                invalido|    870|
|                  valido|2963754|
+------------------------+-------+



# **2. Tratando inconsistências temporais**

## **2.1. Registros de corridas fora do dominio esperado de Janeiro de 2024**

In [14]:
# Removendo registros fora do domínio temporal esperado dos dados

df2 = df.cache()

antes = df2.count()

df2 = df2.filter((F.col("tpep_pickup_datetime") >= "2024-01-01 00:00:00") & (F.col("tpep_pickup_datetime") < "2024-02-01 00:00:00"))

depois = df2.count()

print(f"Linhas antes: {antes}")
print(f"Linhas depois: {depois}")
print(f"Linhas removidas: {antes - depois}")

Linhas antes: 2964624
Linhas depois: 2964606
Linhas removidas: 18


## **2.2. Casos em que os registros de dropoff são anteriores aos de pickup**

Os registros em que `tpep_dropoff_datetime` é anterior a
`tpep_pickup_datetime` foram removidos da base. Essa inconsistência
representa uma violação lógica direta — o fim da corrida não pode
ocorrer antes do seu início — e, dada sua baixa incidência (56
registros, equivalente a **0,0019%** do total de 2.964.624 linhas da
base original), a exclusão não compromete a representatividade nem a
materialidade de nenhuma análise subsequente.

In [15]:
antes = df2.count()

df2 = df2.filter(F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime"))

depois = df2.count()

print(f"Linhas antes: {antes}")
print(f"Linhas depois: {depois}")
print(f"Linhas removidas: {antes - depois}")

Linhas antes: 2964606
Linhas depois: 2964550
Linhas removidas: 56


## **2.3. Outliers de trip_duration_min**

Segundo a [TLC de Nova York](https://www.nyc.gov/site/tlc/about/fatigued-driving-prevention-frequently-asked-questions.page), motoristas de táxi e FHV não podem acumular mais de 10 horas de passenger time (tempo entre o embarque e o desembarque de passageiros) em um período de 24 horas, nem mais de 60 horas em uma semana-calendário (segunda a domingo), sendo o critério utilizado para lidar com outliers a exclusão de registros com duração superior a 10 horas.

In [16]:
antes = df2.count()

df2 = df2.filter(F.col("trip_duration_min") <= 10*60)

depois = df2.count()

print(f"Linhas antes: {antes}")
print(f"Linhas depois: {depois}")
print(f"Linhas removidas: {antes - depois}")

Linhas antes: 2964550
Linhas depois: 2962899
Linhas removidas: 1651


# **3. Tratando inconsistências monetárias**

## **3.1. Validação Matemática da Equação do total_amount**

In [17]:
df3 = df2.cache()

# Cria novas colunas monetárias com vazio substituído por zero
componentes = ["fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
               "improvement_surcharge", "congestion_surcharge", "Airport_fee"]

for c in componentes:
    df3 = df3.withColumn(c + "_sem_vazio", F.coalesce(F.col(c), F.lit(0)))

# Soma as 8 colunas "sem vazio", linha por linha
colunas_sem_vazio = [c + "_sem_vazio" for c in componentes]

df3 = df3.withColumn(
    "calculated_total",
    sum(F.col(c) for c in colunas_sem_vazio)
)

# Arredonda para 2 casas decimais
df3 = df3.withColumn("calculated_total", F.round(F.col("calculated_total"), 2))

# Diferença entre o total já existente e o total recalculado
df3 = df3.withColumn(
    "total_diff",
    F.round(F.col("total_amount") - F.col("calculated_total"), 2)
)

# Remove as colunas auxiliares, que só serviam para o cálculo
for c in componentes:
    df3 = df3.drop(c + "_sem_vazio")

# --- Condições de correção ---
cond1 = (F.col("total_diff") == -2.5) & (F.col("congestion_surcharge") == 2.5)
cond2 = (F.col("total_diff") == -4.25) & (F.col("congestion_surcharge") == 2.5) & (F.col("Airport_fee") == 1.75)
cond3 = (F.col("total_diff") == -1.75) & (F.col("Airport_fee") == 1.75)
cond4 = F.col("total_diff") == 2.5  # Flex Fare, congestion_surcharge nulo

# Quantificação antes de corrigir (para checagem)
print(f"Caso 1 (-2.5, congestion=+2.5): {df3.filter(cond1).count()}")
print(f"Caso 2 (-4.25, congestion=+2.5 e airport=+1.75): {df3.filter(cond2).count()}")
print(f"Caso 3 (-1.75, airport=+1.75): {df3.filter(cond3).count()}")
print(f"Caso 4 (+2.5, congestion nulo): {df3.filter(cond4).count()}")

# --- Correção 1, 2, 3: total_amount passa a ser a soma dos componentes ---
df3 = df3.withColumn(
    "total_amount",
    F.when(cond1 | cond2 | cond3, F.col("calculated_total"))
     .otherwise(F.col("total_amount"))
)

# --- Correção 4: congestion_surcharge nulo recebe 2.5 diretamente ---
df3 = df3.withColumn(
    "congestion_surcharge",
    F.when(F.col("congestion_surcharge").isNull() & cond4, 2.5)
     .otherwise(F.col("congestion_surcharge"))
)

# --- Recalcula calculated_total e total_diff, depois das correções ---
for c in componentes:
    df3 = df3.withColumn(c + "_sem_vazio", F.coalesce(F.col(c), F.lit(0)))

df3 = df3.withColumn(
    "calculated_total",
    F.round(sum(F.col(c + "_sem_vazio") for c in componentes), 2)
)

for c in componentes:
    df3 = df3.drop(c + "_sem_vazio")

df3 = df3.withColumn("total_diff", F.round(F.col("total_amount") - F.col("calculated_total"), 2))

# --- Deleta os demais casos com divergência ainda presente ---
antes = df3.count()
df3 = df3.filter(F.abs(F.col("total_diff")) <= 0.02)
depois = df3.count()

print(f"\nLinhas antes da exclusão final: {antes}")
print(f"Linhas depois da exclusão final: {depois}")
print(f"Linhas removidas: {antes - depois}")

Caso 1 (-2.5, congestion=+2.5): 590940
Caso 2 (-4.25, congestion=+2.5 e airport=+1.75): 22451
Caso 3 (-1.75, airport=+1.75): 18149
Caso 4 (+2.5, congestion nulo): 129319

Linhas antes da exclusão final: 2962899
Linhas depois da exclusão final: 2957703
Linhas removidas: 5196


## **3.2. Análise de outliers e valores anômalos em 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee'** ⚠️

In [18]:
df3_2 = df3.cache()

# GRUPO A — Colunas de valor fixo, sem ambiguidade
# (mta_tax, congestion_surcharge): fora do domínio = erro certo,
# volume ínfimo, sem explicação de regra tarifária. Deleta.

antes = df3_2.count()

dominio_mta_tax = [0.0, 0.5]
dominio_congestion = [0.0, 2.5]

df3_2 = df3_2.filter(
    (F.col("total_amount") < 0) |
    (
        (F.col("mta_tax").isin(dominio_mta_tax)) &
        (F.col("congestion_surcharge").isin(dominio_congestion) | F.col("congestion_surcharge").isNull())
    )
)

depois = df3_2.count()
print(f"[Grupo A] Linhas antes: {antes} | depois: {depois} | removidas: {antes - depois}")

# GRUPO B — Airport_fee - nada a alterar

print("[Grupo B] Airport fee — nenhum tratamento adicional necessário.")

# GRUPO C — improvement_surcharge
# 0,30 tem explicação histórica documentada (taxa vigente de 2015 a 2022) — NÃO é erro sem causa.
# DECISÃO: excluir improvement_surcharge == 0.3 devido ao baixo número de registros (574 - 0.01936%).

antes = df3_2.count()

df3_2 = df3_2.filter(F.col("improvement_surcharge") != 0.3)

depois = df3_2.count()

print(f"[Grupo C] Linhas antes: {antes} | depois: {depois} | removidas: {antes - depois}")

# GRUPO D — fare_amount
# Segmentado por RatecodeID (regras tarifárias diferentes).
# Isenções propositais:
#   - Estornos          -> tratados na seção 3.4
#   - Flex Fare         -> lógica de preço própria, não decomponível
#   - trip_distance = 0 -> classificado na seção 4.2
# CORTE SUPERIOR (200 vs 500): DECISÃO PENDENTE de equipe.

antes = df3_2.count()

df3_2 = df3_2.filter(F.col("fare_amount") <= 500.0)

depois = df3_2.count()

print(f"[Grupo D] Linhas antes: {antes} | depois: {depois} | removidas: {antes - depois}")

# GRUPO E — extra
# Nada a alterar

print("[Grupo E] extra — nenhum tratamento adicional necessário.")

# GRUPO F — tip_amount
# Zero em pagamento dinheiro é esperado (regra de negócio).
# Eliminar registros onde tip_amount é no máximo 3x maior que fare_amount, fare_amount > 0 e tip_amount > 100

antes = df3_2.count()

df3_2 = df3_2.filter(
    ~((F.col("fare_amount") > 0) & (F.col("tip_amount") > 100) & (F.col("tip_amount") > 3 * F.col("fare_amount")))
)

depois = df3_2.count()
print(f"[Grupo F] Linhas antes: {antes} | depois: {depois} | removidas: {antes - depois}")

# GRUPO G — tolls_amount
# 92,9% zero é normal; negativos -> 3.4; valores altos legítimos. Nada a alterar.

print("[Grupo G] tolls_amount — nenhum tratamento adicional necessário.")

[Grupo A] Linhas antes: 2957703 | depois: 2957694 | removidas: 9
[Grupo B] Airport fee — nenhum tratamento adicional necessário.
[Grupo C] Linhas antes: 2957694 | depois: 2957168 | removidas: 526
[Grupo D] Linhas antes: 2957168 | depois: 2957123 | removidas: 45
[Grupo E] extra — nenhum tratamento adicional necessário.
[Grupo F] Linhas antes: 2957123 | depois: 2957105 | removidas: 18
[Grupo G] tolls_amount — nenhum tratamento adicional necessário.


## **3.3. Linhas com total_amount não_negativo e algum registro monetário negativo**⚠️

In [19]:
# Removendo registros com total_amount não-negativo e algum registro monetário negativo

antes = df3_2.count()

df3_2 = df3_2.filter(~((F.col("total_amount") >= 0) & (F.col("fare_amount") < 0)))
df3_2 = df3_2.filter(~((F.col("total_amount") >= 0) & (F.col("extra") < 0)))

depois = df3_2.count()

print(f"Linhas antes: {antes}")
print(f"Linhas depois: {depois}")
print(f"Linhas removidas: {antes - depois}")

Linhas antes: 2957105
Linhas depois: 2955041
Linhas removidas: 2064


## **3.4. Análise de registros negativos em total amount ⚠️**

In [20]:
nao_monetarias = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID",
                   "VendorID", "passenger_count", "trip_distance", "RatecodeID",
                   "store_and_fwd_flag", "payment_type"]

monetarias = ["fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
              "improvement_surcharge", "total_amount", "congestion_surcharge", "Airport_fee"]

# Lado negativo: total_amount < 0 E todas as monetárias <= 0
condicao_todas_nao_positivas = None
for c in monetarias:
    cond = F.col(c) <= 0
    condicao_todas_nao_positivas = cond if condicao_todas_nao_positivas is None else (condicao_todas_nao_positivas & cond)

neg = df3_2.filter((F.col("total_amount") < 0) & condicao_todas_nao_positivas)

# Lado positivo: total_amount >= 0 E todas as monetárias >= 0
condicao_todas_nao_negativas = None
for c in monetarias:
    cond = F.col(c) >= 0
    condicao_todas_nao_negativas = cond if condicao_todas_nao_negativas is None else (condicao_todas_nao_negativas & cond)

pos = df3_2.filter((F.col("total_amount") >= 0) & condicao_todas_nao_negativas)

# --- Identificação dos pares (não-monetárias iguais, monetárias invertidas) ---
neg_invertido = neg.select(
    *nao_monetarias,
    *[(-F.col(c)).alias(c) for c in monetarias]
)

pares_encontrados = neg_invertido.join(pos, on=nao_monetarias + monetarias, how="inner")

# Reconstitui o sinal original (negativo), para saber quais linhas remover
pares_negativos_para_remover = pares_encontrados.select(
    *nao_monetarias,
    *[(-F.col(c)).alias(c) for c in monetarias]
)

qtd_pares = pares_negativos_para_remover.count()
print(f"Pares de estorno identificados (negativo + positivo): {qtd_pares}")

# --- Remoção APENAS da metade negativa de cada par ---
antes = df3_2.count()

df3_2 = df3_2.join(pares_negativos_para_remover, on=nao_monetarias + monetarias, how="left_anti")

depois_pares = df3_2.count()
print(f"Linhas removidas (negativos com par positivo): {antes - depois_pares}")

# --- Exclusão dos órfãos ---
df3_2 = df3_2.filter(F.col("total_amount") >= 0)

depois_final = df3_2.count()
print(f"\nLinhas antes (início da seção 3.4): {antes}")
print(f"Linhas depois (após remoção de pares e órfãos): {depois_final}")
print(f"Total removido: {antes - depois_final}")

Pares de estorno identificados (negativo + positivo): 30570
Linhas removidas (negativos com par positivo): 30570

Linhas antes (início da seção 3.4): 2955041
Linhas depois (após remoção de pares e órfãos): 2919645
Total removido: 35396


# **4. Tratando inconsistências em deslocamento**

## **4.1. Velocidade Média**

In [21]:
df4 = df3_2.cache()

antes = df4.count()

# NOTA: trip_duration_min e average_speed_mph já foram criadas na seção 0.
# Este filtro NÃO exige trip_distance > 0 — os casos de distância zero são
# classificados na seção 4.2, que distingue os válidos por regra tarifária
# (Flex Fare, JFK<->Manhattan, Negotiated fare) dos inválidos.

df4 = df4.filter(
    (F.col("trip_duration_min") > 0.0) &
    (F.col("average_speed_mph").isNull() | (F.col("average_speed_mph") <= 80.0))
)

depois = df4.count()

print(f"Linhas antes: {antes}")
print(f"Linhas depois: {depois}")
print(f"Linhas removidas: {antes - depois}")

Linhas antes: 2919645
Linhas depois: 2917791
Linhas removidas: 1854


## **4.2. trip_distance == 0**

In [22]:
# JOIN com zonas (necessário para JFK-Manhattan)
zonas1 = zone_lookup.select(
    F.col("LocationID").alias("PULocationID"),
    F.col("Zone").alias("PU_Zone"),
    F.col("Borough").alias("PU_Borough")
)
zonas2 = zone_lookup.select(
    F.col("LocationID").alias("DOLocationID"),
    F.col("Zone").alias("DO_Zone"),
    F.col("Borough").alias("DO_Borough")
)
df4 = df4.join(zonas1, "PULocationID").join(zonas2, "DOLocationID")

# As 3 condições onde o regime tarifário já explica trip_distance == 0
cond_flex_fare = F.col("payment_type") == 0
cond_jfk_manhattan = (F.col("RatecodeID") == "2") & (
    ((F.col("PU_Zone") == "JFK Airport") & (F.col("DO_Borough") == "Manhattan")) |
    ((F.col("PU_Borough") == "Manhattan") & (F.col("DO_Zone") == "JFK Airport"))
)
cond_negociada = F.col("RatecodeID") == "5"
casos_validos = cond_flex_fare | cond_jfk_manhattan | cond_negociada

# Evidência de que a corrida ocorreu, mesmo com trip_distance == 0:
# ou o regime já explica (casos_validos), ou PU != DO mostra deslocamento real
evidencia_de_corrida = casos_validos | (F.col("PULocationID") != F.col("DOLocationID"))

df4 = df4.withColumn(
    "status_trip_distance",
    F.when(F.col("trip_distance") != 0, "valido")
     .when(evidencia_de_corrida, "invalido")   # 0 não representa a distância real
     .otherwise("deletar")                       # sem evidência de que a corrida ocorreu
)

df4.groupBy("status_trip_distance").count().orderBy(F.desc("count")).show()

# Remove só as linhas sem evidência de corrida
antes = df4.count()
df4 = df4.filter(F.col("status_trip_distance") != "deletar")
depois = df4.count()

print(f"Linhas antes: {antes}")
print(f"Linhas depois: {depois}")
print(f"Linhas removidas: {antes - depois}")

+--------------------+-------+
|status_trip_distance|  count|
+--------------------+-------+
|              valido|2862514|
|            invalido|  42712|
|             deletar|  12565|
+--------------------+-------+

Linhas antes: 2917791
Linhas depois: 2905226
Linhas removidas: 12565


# **5. passenger_count**

## **5.1. passenger_count >= 7**


In [23]:
df5 = df4.cache()

antes = df5.count()

df5 = df5.filter(F.col("passenger_count").isNull() | (F.col("passenger_count") < 7))

depois = df5.count()

print(f"Linhas antes: {antes}")
print(f"Linhas depois: {depois}")
print(f"Linhas removidas: {antes - depois}")

Linhas antes: 2905226
Linhas depois: 2905179
Linhas removidas: 47


## **5.2. passenger_count = 0**

In [24]:
antes = df5.count()

df5 = df5.withColumn(
    "passenger_count_status",
    F.when(F.col("passenger_count").isNull(), "invalido")
     .when(F.col("passenger_count") == 0, "invalido")
     .otherwise("valido")
)

depois = df5.count()

print(f"Linhas antes: {antes}")
print(f"Linhas depois: {depois}")  # deve ser igual — nenhuma linha é removida aqui

df5.groupBy("passenger_count_status").count().orderBy(F.desc("count")).show(15)

Linhas antes: 2905179
Linhas depois: 2905179
+----------------------+-------+
|passenger_count_status|  count|
+----------------------+-------+
|                valido|2737506|
|              invalido| 167673|
+----------------------+-------+



In [25]:
df5.show()

+------------+------------+--------------------+---------------------+--------+---------------+-------------+----------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+------------------+------------------+----------------------+---------------------------+------------------+------------------------+----------------+----------+--------------------+----------+--------------------+----------+--------------------+
|DOLocationID|PULocationID|tpep_pickup_datetime|tpep_dropoff_datetime|VendorID|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee| trip_duration_min| average_speed_mph|     cost_per_mile|passenger_count_status|congestion_surcharge_status|Airport_fee_status|average_speed_mph_status|calculated_total|total_diff|             P

# **6. Últimos ajustes**

In [26]:
df_silver = df5.drop(
    "calculated_total", "total_diff",
    "PU_Zone", "PU_Borough", "DO_Zone", "DO_Borough"
)

df_silver.show()

+------------+------------+--------------------+---------------------+--------+---------------+-------------+----------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+------------------+------------------+----------------------+---------------------------+------------------+------------------------+--------------------+
|DOLocationID|PULocationID|tpep_pickup_datetime|tpep_dropoff_datetime|VendorID|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee| trip_duration_min| average_speed_mph|     cost_per_mile|passenger_count_status|congestion_surcharge_status|Airport_fee_status|average_speed_mph_status|status_trip_distance|
+------------+------------+--------------------+---------------------+--------+---------------+-------------+---

# **7. Load Camada Silver**

In [30]:
# Monta a Silver final já sem as colunas de andaime
df_silver = df5.drop(
    "calculated_total", "total_diff",
    "PU_Zone", "PU_Borough", "DO_Zone", "DO_Borough"
)

diretorio_raiz = Path(os.getcwd())
silver_dir = diretorio_raiz / "data" / "2_Silver" / "yellow_trips_silver"
silver_dir.mkdir(parents=True, exist_ok=True)
silver_path = silver_dir / "silver.parquet"

# Grava um único Parquet dentro da pasta yellow_trips_silver (sem winutils)
df_silver.toPandas().to_parquet(
    str(silver_path),
    index=False,
    coerce_timestamps="us",          # grava timestamp em microssegundos (o que o Spark lê)
    allow_truncated_timestamps=True  # permite descartar a precisão de nanos sem erro
)

print(f"✅ Silver salva em: {silver_path}")
print("Linhas:", df_silver.count(), "| Colunas:", len(df_silver.columns))

✅ Silver salva em: c:\Users\danil\OneDrive\Documentos\Projeto LED\data\2_Silver\yellow_trips_silver\silver.parquet
Linhas: 2905179 | Colunas: 27


In [31]:
df_silver.printSchema()
print("Total de linhas na Prata:", df_silver.count())

root
 |-- DOLocationID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- VendorID: integer (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- trip_duration_min: double (nullable = true)
 |-- average_speed_mph: double (nullable = true)
 |-- cost_per_mile: double